In [2]:
# Simple RDMA Pipeline Test for Jupyter Notebook

import numpy as np
import torch
from rdma.rd.extractor import RDMAExtractor
from rdma.rd.verifier import RDMAVerifier
from rdma.rd.matcher import RDMAMatcher
from rdma.utils.llm_client import LocalLLMClient
from rdma.utils.embedding import EmbeddingsManager

# Sample clinical texts
sample_texts = [
    "Patient presents with signs of Fabry disease including angiokeratomas and acroparesthesias. "
    "Family history is significant for renal failure in maternal uncle. "
    "No evidence of Gaucher disease. Labs show elevated globotriaosylceramide levels.",
    
    "43-year-old female with SLE and progressive dyspnea. Workup revealed PAH with RVSP of 68 mmHg. "
    "Diagnosis of CTD-PAH was made. Also noted was mild achalasia and Raynaud's phenomenon, "
    "raising suspicion for limited scleroderma (CREST syndrome)."
]

# Initialize LLM client
device = "cuda:0" if torch.cuda.is_available() else "cpu"
llm_client = LocalLLMClient(
    model_type="mistral24b",  # Using a smaller model for faster testing
    device=device,
    temperature=0.1
)

# Initialize embedding manager
embeddings_file = "data/vector_stores/rd_orpha_medembed.npy"
embedding_manager = EmbeddingsManager(
    model_type="sentence_transformer",
    model_name="abhinand/MedEmbed-small-v0.1",
    device=device
)

# Load embedded documents
embedded_documents = np.load(embeddings_file, allow_pickle=True)

# Initialize extractor
extractor = RDMAExtractor(
    llm_client=llm_client,
    extraction_method="retrieval",
    embedding_manager=embedding_manager,
    embedded_documents=embedded_documents,
    window_size=1,
    top_k=5,
    min_sentence_size=200,
    debug=True
)

# Extract entities from sample texts
extraction_results = {}
for i, text in enumerate(sample_texts):
    print(f"\nExtracting entities from text {i+1}:")
    entities_with_contexts = extractor.extract_from_text(text)
    print(f"Found {len(entities_with_contexts)} potential entities")
    
    # Display some extracted entities
    for j, entity in enumerate(entities_with_contexts[:2]):
        print(f"  Entity {j+1}: {entity.get('entity', '')}")
        
    # Store for next step
    extraction_results[f"text_{i+1}"] = {
        "clinical_text": text,
        "entities_with_contexts": entities_with_contexts,
    }

# Initialize verifier
verifier = RDMAVerifier(
    llm_client=llm_client,
    embedding_manager=embedding_manager,
    embedded_documents=embedded_documents,
    verifier_type="multi_stage",
    min_context_length=5,
    debug=True
)

# Verify entities
verification_results = verifier.verify_from_json(extraction_results)

# Display verification results
for text_id, result in verification_results.items():
    verified_entities = result.get("verified_rare_diseases", [])
    print(f"\n{text_id}: Verified {len(verified_entities)} entities as rare diseases")
    
    # Display some verified entities
    for j, entity in enumerate(verified_entities[:2]):
        print(f"  Verified entity {j+1}: {entity.get('entity', '')}")

# Initialize matcher
matcher = RDMAMatcher(
    llm_client=llm_client,
    embedding_manager=embedding_manager,
    embedded_documents=embedded_documents,
    top_k=5,
    debug=True
)

# Match entities to ORPHA codes
matching_results = matcher.match_from_json(verification_results)

# Display matching results
for text_id, result in matching_results.items():
    matched_entities = result.get("matched_diseases", [])
    print(f"\n{text_id}: Matched {len(matched_entities)} entities to ORPHA codes")
    
    # Display some matched entities
    for j, entity in enumerate(matched_entities[:2]):
        print(f"  Entity: {entity.get('entity', '')}")
        print(f"  Matched to: {entity.get('rd_term', '')} (ORPHA:{entity.get('orpha_id', '')})")
        print(f"  Method: {entity.get('match_method', '')}, Confidence: {entity.get('confidence_score', 0):.2f}")

Initialized ModelLoader with cache directory: /shared/rsaas/jw3/rare_disease/model_cache
Loading LLM!
Device configuration: cuda:0
Using device map: {'': 'cuda:0'}
Loading 70B model with quantization: mistral24b
Generated cache path: /shared/rsaas/jw3/rare_disease/model_cache/OpenBioLLM-Llama3-70B_4bit_nf4
Valid cache found at /shared/rsaas/jw3/rare_disease/model_cache/OpenBioLLM-Llama3-70B_4bit_nf4
Loading cached quantized model from /shared/rsaas/jw3/rare_disease/model_cache/OpenBioLLM-Llama3-70B_4bit_nf4


/home/johnwu3/miniconda3/envs/hporag/lib/python3.10/site-packages/transformers/quantizers/auto.py:206: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Device set to use cuda:0


Hello! I'm here to help answer any questions you may have. Please go ahead and ask your question.  Note: Since this is an open-ended question, I will provide a general response without the need for format constraints.
Loading model...
Model type: sentence_transformer
Model name: abhinand/MedEmbed-small-v0.1
Device: cuda:0
Initializing SentenceTransformer with model: abhinand/MedEmbed-small-v0.1 on device: cuda:0
Model successfully moved to cuda:0
Verifying model by embedding sample text...
Model initialized successfully. Embedding dimension: 384
12:07:39 | Initializing retrieval-enhanced entity extractor

Extracting entities from text 1:
After merging: Processing 1 chunks instead of 4 raw sentences
12:07:45 | Extracted 5 potential rare disease entities
12:07:45 | Extracting contexts for 5 entities
12:07:45 | Extracted 4 sentences from text
12:07:45 |   Entity: 'Fabry disease'
12:07:45 |     Context: 'Patient presents with signs of Fabry disease including angiokeratomas and acroparesthe

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


12:08:01 |     No close ORPHA match found via string or LLM matching for 'elevated globotriaosylceramide levels'
12:08:01 |   'elevated globotriaosylceramide levels' is not a rare disease
12:08:01 | Identified 3 verified rare diseases
12:08:01 | Processing case 2/2 (ID: text_2)
12:08:01 |   Processing 7 raw entities
12:08:01 | Verifying 7 entities after filtering
12:08:01 | Processing batch of 7 entities
12:08:01 | Found 7 unique entity-context pairs
12:08:01 | Processing entity: 'SLE'
12:08:01 |   Verifying if 'SLE' is a rare disease via multi-step process
12:08:01 |     Exact ORPHA match found: 'SLE' matches 'sle' (Orpha:536)
12:08:03 |     Multi-step verification: 'SLE' is a rare disease
12:08:03 |   'SLE' is a rare disease
12:08:03 | Processing entity: 'PAH'
12:08:03 |   Verifying if 'PAH' is a rare disease via multi-step process
12:08:03 |     Exact ORPHA match found: 'PAH' matches 'pah' (Orpha:182090)
12:08:04 |     Multi-step verification: 'PAH' is a rare disease
12:08:04 |   'P